# 창원국가산단 산업·고용 전환진단 및 지원연계 모형

모형 명칭: **ELECTRE TRI-B의 경계 프로파일과 할당 절차를 적용한 다기준 경계분류 시범모형**

이 노트북은 `src/model`의 함수를 호출해 결과를 **표시만** 한다. 파생지표·품질 플래그·파라미터 검증·ELECTRE 계산·진단카드 생성 로직은 모두 `src/model`에 있고, 기존 `03_q1_q2_q3_integrated_analysis.ipynb`와 `src/eda`는 수정하지 않는다.

| 구분 | 내용 |
|---|---|
| 데이터가 직접 보여주는 것 | 생산·고용 수준과 YoY, 고용감소 인원, 고용 YoY 기준하회 기간, 업체 수 변화, PPI 후보 적용 시 방향 변화 |
| 모형이 부여하는 것 | 시나리오별 점검단계(OBSERVE·CHECK·PRIORITY)와 판정불가(UNDETERMINED) |
| 현재 자료로 입증할 수 없는 것 | 고용·생산 변화의 원인, 특정 지원정책의 필요성·효과, 미래 상태가 나타날 확률 |

가중치·경계·lambda·delta_emp·require_employment_evidence가 등록되지 않으면 점검단계를 만들지 않는다. ELECTRE 산출물은 현재 실행 manifest에서 `CURRENT`로 확인된 파일만 읽는다.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'data/processed/kicox/changwon_state_panel.csv').is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError('프로젝트의 data/processed/kicox/changwon_state_panel.csv가 필요합니다.')
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from model import config, manifest, pipeline

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)
print(config.MODEL_TITLE)
print(config.MODEL_ASSIGNMENT_METHOD_NOTE)

ELECTRE TRI-B의 경계 프로파일과 할당 절차를 적용한 다기준 경계분류 시범모형
할당 절차는 pessimistic assignment이다. 높은 경계 b2부터 비교해 outrank하면 우선점검, 그렇지 않고 b1을 outrank하면 추가확인, 둘 다 아니면 관찰로 배정한다.


## 0. 실행과 현재 실행 manifest

`pipeline.run()`이 입력 패널·보조지표·검증 보고서·진단카드를 계산하고 실행별 디렉터리(`outputs/runs/<run_id>/`)와 canonical 위치에 저장한다. `outputs/current_run_manifest.json`이 이번 실행에서 쓴 파일만 `CURRENT`로 표시한다.

In [2]:
bundle = pipeline.run(ROOT, write=True)
report = bundle['report']
pv = report['parameter_validation']
ctx = bundle['ctx']
current = manifest.load_manifest(ROOT)
assert current['run_id'] == bundle['run_id'] == report['run_id']

print(f"실행 ID         : {bundle['run_id']}")
print(f"분석기간        : {ctx.quarters[0]} ~ {ctx.latest} ({len(ctx.quarters)}분기)")
print(f"실행 상태       : {report['run_mode']}")
print(f"파라미터 상태   : {pv['effective_status']} (파일: {pv['parameter_file']}, 존재: {pv['parameter_file_present']})")
print(f"기준 패널 해시  : {bundle['provenance']['input_sha256']} / 원본 바이트 {bundle['provenance']['input_sha256_raw_bytes']}")
print(f"fixture 대조    : {report['fixture_check']['status']}")
print(f"표현 점검       : {report['wording_audit']['status']} / passed={report['wording_audit']['passed']}")
display(pd.DataFrame([{'key': k, **{f: v.get(f) for f in ('canonical_path', 'status', 'file_run_id')}}
                      for k, v in current['files'].items()]))

실행 ID         : 20260915T074919114280Z-993960af
분석기간        : 2022Q1 ~ 2026Q2 (18분기)
실행 상태       : BLOCKED_MISSING_PARAMETERS
파라미터 상태   : DRAFT (파일: config/electre_tri_b_params.template.yaml, 존재: False)
기준 패널 해시  : 58499908a6af50344a1362c55c840a58cfd80d4373c7098c4c97a9e0271a573b / 원본 바이트 9eab0a418a59d45db45d451cd60b05233a7a4f1c756a49150fbd3e50c78ddd6a
fixture 대조    : MATCH
표현 점검       : CHECKED / passed=True


,key,canonical_path,status,file_run_id
0,input_panel,data/processed/model/electre_input_panel.csv,CURRENT,None
1,eligibility_audit,outputs/tables/electre_eligibility_audit.csv,CURRENT,None
2,criteria_redundancy,outputs/tables/criteria_redundancy_diagnostics...,CURRENT,None
3,g4_delta_sensitivity,outputs/tables/g4_delta_sensitivity.csv,CURRENT,None
4,qoq_yoy_comparison,outputs/tables/qoq_yoy_comparison.csv,CURRENT,None
5,firm_count_context,outputs/tables/firm_count_context.csv,CURRENT,None
6,parameter_register,outputs/tables/electre_parameter_register.csv,CURRENT,None
7,cards_csv,outputs/tables/diagnostic_cards_latest.csv,CURRENT,None
8,eis_regional_context,outputs/tables/eis_regional_context.csv,CURRENT,None
9,cards_md,outputs/report/diagnostic_cards_latest.md,CURRENT,None


## 1. 기준 패널 검증(V1~V6)

기준 패널은 03 노트북과 같은 `changwon_state_panel.csv`이며, `src/eda/panel.verify_panel()` 검산을 먼저 다시 통과한다.

In [3]:
print(report['base_panel_verification'])
display(pd.DataFrame([{'검증': k, '내용': v['label'], '통과': v['passed']}
                      for k, v in report['panel_validation'].items()]))
v = report['panel_validation']
print(f"V1: {v['V1']['rows']}행, 키 중복 {v['V1']['duplicate_keys']}건, 업종 {v['V1']['industries']}개 × 분기 {v['V1']['quarters']}개")
print(f"V2: 항등식 위반 {v['V2']['violations']}건 / 평가 {v['V2']['rows_evaluated']}행")
print(f"V3: {v['V3']['description']}")
display(pd.DataFrame(v['V4']['counts']).rename(columns={'label': 'threshold 품질 플래그', 'count': '관측 수'}))
print(f"V5: threshold 상향 시 S1~S4 간 직접 이동 {v['V5']['s_to_other_s_moves']}건")
print(f"V6: 판정불가 {v['V6']['unscorable_rows']}행 — {v['V6'].get('note', '')}")

검산: 달력 분기, YoY, 국면, 연속기간, 인접 전환 일치


,검증,내용,통과
0,V1,V1 행 수·기본키 중복,True
1,V2,V2 (g1>0) == (g2>0) == (g4_delta00>=1),True
2,V3,V3 g3(명목 생산감소) 결측,True
3,V4,V4 threshold 품질 플래그 건수,True
4,V5,V5 threshold 상향 시 S1~S4 간 직접 이동,True
5,V6,V6 판정불가 행에서 concordance·점검단계 미생성,True


V1: 180행, 키 중복 0건, 업종 10개 × 분기 18개
V2: 항등식 위반 0건 / 평가 180행
V3: 원천 생산 결측은 2023Q4 10행이며, 이 값이 전년동기 기준으로 사용되는 2024Q4 10행까지 파생 결측을 발생시켜 생산 YoY 결측은 총 20행이다.


,threshold 품질 플래그,관측 수
0,STABLE_2_0,113
1,SENSITIVE_0_5,7
2,SENSITIVE_1_0,12
3,SENSITIVE_2_0,24
4,EXACT_ZERO,4
5,INVALID,20


V5: threshold 상향 시 S1~S4 간 직접 이동 0건
V6: 판정불가 20행 — ELECTRE 미실행 — 입력 패널에 concordance·점검단계 컬럼을 만들지 않음


## 2. 핵심 기준 — 최신분기 관측값

모든 기준은 값이 클수록 먼저 확인할 필요가 커지는 방향이다. g3는 **명목** 생산액 기준이며 PPI 조정값으로 대체하지 않는다. 업종 순서는 최신분기 고용 규모순이며 점검 우선순위가 아니다.

In [4]:
G = config.CRITERION_COLUMNS
panel = bundle['panel']
latest = panel[panel.quarter == ctx.latest].set_index('industry').reindex(ctx.ind_order_emp)
criteria_table = latest[['state', G['g1'], G['g2'], G['g3'], 'g4_delta00', 'g4_delta05', 'g4_delta10',
                         'core_data_status', 'q1_routing_status', 'threshold_flag']].rename(columns={
    'state': 'Q1 상태', G['g1']: config.CRITERION_LABELS['g1'], G['g2']: config.CRITERION_LABELS['g2'],
    G['g3']: config.CRITERION_LABELS['g3'],
    'g4_delta00': f"{config.CRITERION_LABELS['g4']}(δ=0.0)",
    'g4_delta05': f"{config.CRITERION_LABELS['g4']}(δ=0.5)",
    'g4_delta10': f"{config.CRITERION_LABELS['g4']}(δ=1.0)"})
display(criteria_table.round(4))
print(report['structural_observations']['note'])
print(f"완전관측 {report['structural_observations']['complete_rows']}행 중 명목 생산만 감소한 행: "
      f"{report['structural_observations']['production_only_decline_rows']}행")

,Q1 상태,고용감소 절대규모(명),고용감소 상대규모(%),명목 생산감소 정도(%),전년동기 대비 고용 하회 연속분기(δ=0.0),전년동기 대비 고용 하회 연속분기(δ=0.5),전년동기 대비 고용 하회 연속분기(δ=1.0),core_data_status,q1_routing_status,threshold_flag
industry,,,,,,,,,,
기계,S4,3979.0,6.3397,19.5474,2,2,2,COMPLETE,ROUTABLE_S4,STABLE_2_0
전기전자,S2,116.0,0.4129,0.0000,6,0,0,COMPLETE,ROUTABLE_S2,SENSITIVE_0_5
운송장비,S2,129.0,0.7848,0.0000,6,6,0,COMPLETE,ROUTABLE_S2,SENSITIVE_1_0
철강,S2,112.0,1.1427,0.0000,4,4,1,COMPLETE,ROUTABLE_S2,SENSITIVE_2_0
음식료,S2,25.0,3.7879,0.0000,6,6,6,COMPLETE,ROUTABLE_S2,SENSITIVE_1_0
석유화학,S3,0.0,0.0000,6.7298,0,0,0,COMPLETE,ROUTABLE_S3,STABLE_2_0
목재종이,S4,48.0,10.0840,12.8971,2,2,2,COMPLETE,ROUTABLE_S4,STABLE_2_0
비금속,S2,2.0,1.0811,0.0000,18,18,18,COMPLETE,ROUTABLE_S2,SENSITIVE_2_0
기타,S1,0.0,0.0000,0.0000,0,0,0,COMPLETE,ROUTABLE_S1,STABLE_2_0


명목 생산 YoY는 음수이고 고용 YoY는 0 이상인 행이다. 이 행이 경계를 통과하는지는 데이터가 아니라 기준 구성(g1·g2·g4가 모두 고용 차원)과 파라미터 선택(w3, lambda, require_employment_evidence)에 따라 결정된다. 행은 삭제하지 않는다.
완전관측 160행 중 명목 생산만 감소한 행: 16행


## 3. 기준 중복성 진단(파라미터 없이 산출)

완전관측 행의 g1~g4 Pearson·Spearman 상관과 고용감소 행(g1>0)의 g1·g2 상관이다. 상관계수는 원인 관계나 기준 중복을 확정하지 않는다.

In [5]:
print(config.REDUNDANCY_NOTE)
display(bundle['redundancy'][['scope', 'criterion_x', 'criterion_y', 'method', 'correlation', 'n_rows_used',
                              'n_rows_excluded_missing', 'n_rows_excluded_by_scope']].round(4))

상관계수는 두 기준 값이 함께 움직이는 정도를 요약한 기술통계다. 원인 관계나 기준 중복을 확정하지 않으며, 고용이 감소하지 않은 행에서 여러 고용 기준이 함께 0이 되는 구조의 영향을 받는다.


,scope,criterion_x,criterion_y,method,correlation,n_rows_used,n_rows_excluded_missing,n_rows_excluded_by_scope
0,complete_cases,g1,g2,pearson,0.0930,160,20,0
1,complete_cases,g1,g2,spearman,0.8194,160,20,0
2,complete_cases,g1,g3,pearson,-0.0493,160,20,0
3,complete_cases,g1,g3,spearman,0.2725,160,20,0
4,complete_cases,g1,g4,pearson,0.1232,160,20,0
5,complete_cases,g1,g4,spearman,0.7895,160,20,0
6,complete_cases,g2,g3,pearson,0.3384,160,20,0
7,complete_cases,g2,g3,spearman,0.3633,160,20,0
8,complete_cases,g2,g4,pearson,0.4911,160,20,0
9,complete_cases,g2,g4,spearman,0.8531,160,20,0


## 4. 전년동기 대비 고용 하회 연속분기 — δ 민감도와 절단·포화

분석 첫 분기부터 이어진 run은 좌절단이므로 실제 지속기간은 표시값 **이상**이다. 분석창 전체 길이에 도달하면 포화로 표시한다.

In [6]:
display(bundle['cards'].set_index('industry')[[f'{c}_text' for c in config.G4_DELTA_COLUMNS]])

,g4_delta00_text,g4_delta05_text,g4_delta10_text
industry,,,
기계,δ=0.0: 고용 YoY가 기준을 2분기 연속 하회 (최신분기까지 이어진 열린 run),δ=0.5: 고용 YoY가 기준을 2분기 연속 하회 (최신분기까지 이어진 열린 run),δ=1.0: 고용 YoY가 기준을 2분기 연속 하회 (최신분기까지 이어진 열린 run)
전기전자,δ=0.0: 고용 YoY가 기준을 6분기 연속 하회 (최신분기까지 이어진 열린 run),δ=0.5: 고용 YoY가 기준을 하회하지 않음(0분기),δ=1.0: 고용 YoY가 기준을 하회하지 않음(0분기)
운송장비,δ=0.0: 고용 YoY가 기준을 6분기 연속 하회 (최신분기까지 이어진 열린 run),δ=0.5: 고용 YoY가 기준을 6분기 연속 하회 (최신분기까지 이어진 열린 run),δ=1.0: 고용 YoY가 기준을 하회하지 않음(0분기)
철강,δ=0.0: 고용 YoY가 기준을 4분기 연속 하회 (최신분기까지 이어진 열린 run),δ=0.5: 고용 YoY가 기준을 4분기 연속 하회 (최신분기까지 이어진 열린 run),δ=1.0: 고용 YoY가 기준을 1분기 연속 하회 (최신분기까지 이어진 열린 run)
음식료,δ=0.0: 고용 YoY가 기준을 6분기 연속 하회 (최신분기까지 이어진 열린 run),δ=0.5: 고용 YoY가 기준을 6분기 연속 하회 (최신분기까지 이어진 열린 run),δ=1.0: 고용 YoY가 기준을 6분기 연속 하회 (최신분기까지 이어진 열린 run)
석유화학,δ=0.0: 고용 YoY가 기준을 하회하지 않음(0분기),δ=0.5: 고용 YoY가 기준을 하회하지 않음(0분기),δ=1.0: 고용 YoY가 기준을 하회하지 않음(0분기)
목재종이,δ=0.0: 고용 YoY가 기준을 2분기 연속 하회 (최신분기까지 이어진 열린 run),δ=0.5: 고용 YoY가 기준을 2분기 연속 하회 (최신분기까지 이어진 열린 run),δ=1.0: 고용 YoY가 기준을 2분기 연속 하회 (최신분기까지 이어진 열린 run)
비금속,δ=0.0: 고용 YoY가 기준을 최소 18분기 연속 하회 (분석 시작분기부터 이어...,δ=0.5: 고용 YoY가 기준을 최소 18분기 연속 하회 (분석 시작분기부터 이어...,δ=1.0: 고용 YoY가 기준을 최소 18분기 연속 하회 (분석 시작분기부터 이어...
기타,δ=0.0: 고용 YoY가 기준을 하회하지 않음(0분기),δ=0.5: 고용 YoY가 기준을 하회하지 않음(0분기),δ=1.0: 고용 YoY가 기준을 하회하지 않음(0분기)


## 5. QoQ·업체 수 보조지표

업체 수 관련 값은 ELECTRE 기준이나 점검단계에 쓰지 않는다. 가동업체 10개 이하 표시는 소규모 집계 주의 표시다. 업체 수 변화를 고용 변화의 원인으로 해석하지 않는다.

In [7]:
display(latest[['employment', 'emp_delta', 'employment_yoy', 'emp_qoq_delta', 'qoq_recovery_while_yoy_below',
                'firms_op', 'firm_count_delta_qoq', 'firm_count_delta_yoy', 'firm_count_changed_yoy',
                'emp_per_firm', 'small_firm_count_flag', 'op_rate_used', 'op_rate_used_source',
                'op_rate_used_yoy_pp', 'op_rate_yoy_basis']].round(2))
print(config.SMALL_FIRM_CAUTION)

,employment,emp_delta,employment_yoy,emp_qoq_delta,qoq_recovery_while_yoy_below,firms_op,firm_count_delta_qoq,firm_count_delta_yoy,firm_count_changed_yoy,emp_per_firm,small_firm_count_flag,op_rate_used,op_rate_used_source,op_rate_used_yoy_pp,op_rate_yoy_basis
industry,,,,,,,,,,,,,,,
기계,58784.0,-3979.0,-6.34,-104.0,False,1458.0,-4.0,5.0,True,40.32,False,80.61,op_rate_official,-4.89,SAME_SOURCE
전기전자,27979.0,-116.0,-0.41,35.0,True,590.0,1.0,19.0,True,47.42,False,76.65,op_rate_official,-0.38,SAME_SOURCE
운송장비,16308.0,-129.0,-0.78,-67.0,False,245.0,-1.0,1.0,True,66.56,False,92.28,op_rate_official,0.97,SAME_SOURCE
철강,9689.0,-112.0,-1.14,14.0,True,74.0,-1.0,-1.0,True,130.93,False,63.68,op_rate_official,0.24,SAME_SOURCE
음식료,635.0,-25.0,-3.79,7.0,True,4.0,0.0,0.0,False,158.75,True,92.30,op_rate_official,-1.43,SAME_SOURCE
석유화학,619.0,54.0,9.56,-3.0,False,45.0,2.0,4.0,True,13.76,False,76.31,op_rate_official,-3.32,SAME_SOURCE
목재종이,428.0,-48.0,-10.08,0.0,False,44.0,-2.0,-1.0,True,9.73,False,74.71,op_rate_official,6.01,SAME_SOURCE
비금속,183.0,-2.0,-1.08,1.0,True,4.0,-1.0,-1.0,True,45.75,True,57.30,op_rate_official,14.11,SAME_SOURCE
기타,171.0,25.0,17.12,2.0,False,25.0,-1.0,-2.0,True,6.84,False,69.14,op_rate_official,1.61,SAME_SOURCE


가동업체 수가 적어 개별 업체 수준 변화가 업종 집계에 크게 반영될 수 있음. 현재 집계자료만으로 업체 수 변화가 고용 변화의 원인이라고 판단할 수 없음.


## 6. PPI 후보 적용 시 생산 방향 민감도

PPI 조정 생산 YoY = ((1 + 명목 생산 YoY/100) / (1 + PPI YoY/100) − 1) × 100. 매핑 확정 상태는 원자료 confirmed 값으로 표시하며 공식 점검등급이 아니다. 복수 후보는 대안 상태를 집합으로 표시한다.

In [8]:
display(pd.DataFrame(report['ppi_latest']).set_index('industry').round(3))
cross = report['crosschecks']
display(pd.DataFrame([{'교차검산': k, '통과': v.get('passed')} for k, v in cross.items()]))
print('매핑 확정 상태(업종 수):', cross['ppi_mapping_confirmation']['status_counts_by_industry'])
print('전 기간 PPI 방향 상태 건수:', cross['ppi_sensitivity_full_period'].get('model_direction_status_counts'))

,state,production_yoy,ppi_candidate_n,ppi_yoy_min,ppi_yoy_max,ppi_adjusted_prod_yoy_lower,ppi_adjusted_prod_yoy_upper,ppi_direction_status,ppi_alt_state_set,ppi_alt_state_all_same,ppi_mapping_review_type,ppi_mapping_confirmed,ppi_mapping_confirmation_status
industry,,,,,,,,,,,,,
기계,S4,-19.547,1.0,1.258,1.258,-20.547,-20.547,SAME_ALL,S4,True,단일 후보·근접 분류,False,NOT_CONFIRMED
기타,S1,3.033,NaN,NaN,NaN,NaN,NaN,NOT_COMPARABLE,NaN,None,매핑 불가,False,NOT_CONFIRMED
목재종이,S4,-12.897,1.0,2.824,2.824,-15.289,-15.289,SAME_ALL,S4,True,단일 후보·근접 분류,False,NOT_CONFIRMED
비금속,S2,45.671,1.0,3.415,3.415,40.861,40.861,SAME_ALL,S2,True,단일 후보·근접 분류,False,NOT_CONFIRMED
석유화학,S3,-6.730,2.0,19.155,71.255,-45.537,-21.724,SAME_ALL,S3,True,복수 후보·범위 제시,False,NOT_CONFIRMED
섬유의복,N,-12.320,1.0,3.360,3.360,-15.170,-15.170,SAME_ALL,N,True,단일 후보·분류 선택,False,NOT_CONFIRMED
운송장비,S2,19.643,1.0,2.050,2.050,17.240,17.240,SAME_ALL,S2,True,단일 후보·범위 주의,False,NOT_CONFIRMED
음식료,S2,0.873,1.0,1.666,1.666,-0.779,-0.779,OPPOSITE_ALL,S4,True,단일 후보·근접 분류,False,NOT_CONFIRMED
전기전자,S2,2.698,2.0,8.291,20.984,-15.114,-5.165,OPPOSITE_ALL,S4,True,복수 후보·범위 제시,False,NOT_CONFIRMED


,교차검산,통과
0,ppi_ratio_formula,True
1,ppi_sensitivity_full_period,True
2,aux_summary_latest,True
3,ppi_mapping_confirmation,True
4,ppi_mapping_all_unconfirmed,True
5,residual_category_not_comparable,True
6,eis_isolation,True
7,firm_count_isolation,True


매핑 확정 상태(업종 수): {'NOT_CONFIRMED': 10}
전 기간 PPI 방향 상태 건수: {'CANDIDATE_DEPENDENT': 6, 'OPPOSITE_ALL': 13, 'SAME_ALL': 125}


## 7. 파라미터 등록 상태·단계별 후속 행동 검증과 사람의 결정이 필요한 항목

In [9]:
print(f"실행 상태: {pv['run_mode']} / 오류 {len(pv['errors'])}건 / 경고 {len(pv['warnings'])}건")
print(f"비어 있는 파라미터 필드 {len(pv['missing_parameter_fields'])}개")
display(bundle['register'][['scenario_id', 'parameter', 'criterion', 'boundary', 'value', 'value_status']])
sa = report['stage_actions']
print(f"단계별 후속 행동: {sa['status']} (출처 {sa['source']}, 오류 {len(sa['errors'])}건, 비어 있는 필드 {len(sa['missing_fields'])}개)")
print('사람의 결정이 필요한 항목:', ', '.join(report['policy_decisions_required']['decisions']))

실행 상태: BLOCKED_MISSING_PARAMETERS / 오류 0건 / 경고 0건
비어 있는 파라미터 필드 45개


,scenario_id,parameter,criterion,boundary,value,value_status
0,기준,weight,g1,NaN,None,MISSING
1,기준,weight,g2,NaN,None,MISSING
2,기준,weight,g3,NaN,None,MISSING
3,기준,weight,g4,NaN,None,MISSING
4,기준,profile,g1,b1,None,MISSING
5,기준,profile,g2,b1,None,MISSING
6,기준,profile,g3,b1,None,MISSING
7,기준,profile,g4,b1,None,MISSING
8,기준,profile,g1,b2,None,MISSING
9,기준,profile,g2,b2,None,MISSING


단계별 후속 행동: POLICY_DECISION_REQUIRED (출처 config/stage_actions.template.yaml, 오류 0건, 비어 있는 필드 16개)
사람의 결정이 필요한 항목: 세 시나리오별 weights, b1·b2, lambda, delta_emp, require_employment_evidence, 단계별 담당 역할, 처리기한, 공식 승인 여부, 단순 개수규칙 최종 채택 여부


## 8. 시나리오별 점검단계(등록된 파라미터가 있을 때만)

ELECTRE 산출물은 `manifest.load_current_table()`로만 읽는다. 현재 실행이 만들지 않은 과거 파일은 읽지 않는다. `SCENARIO_REGISTERED`에서는 단일 공식 등급을 만들지 않고, 단순 개수규칙 비교는 결과만 제시한다.

In [10]:
status = {k: manifest.current_table_status(ROOT, k) for k in config.PARAMETER_OUTPUTS}
display(pd.Series(status, name='현재 실행 기준 상태').to_frame())
assignments_current = manifest.load_current_table(ROOT, 'assignments')
if assignments_current is None:
    print(f"점검단계 미산출 — {report['run_mode']}. 가중치·경계·lambda를 임의로 채우지 않았다.")
    for item in report['stale_parameter_outputs_present']:
        print(f"현재 결과가 아닌 과거 파일(읽지 않음): {item['canonical_path']}")
else:
    e = report['electre']
    print(e['concordance_note'])
    summary_current = manifest.load_current_table(ROOT, 'scenario_summary')
    display(summary_current[summary_current.quarter == ctx.latest])
    display(pd.DataFrame(e['simple_rule_observed']['by_scenario_boundary']))
    print(f"관측자료 불일치 합집합: {e['simple_rule_observed']['union_n_rows']}행")
    display(pd.DataFrame(e['simple_rule_function_level']))
    display(manifest.load_current_table(ROOT, 'criterion_removal_sensitivity'))
    display(manifest.load_current_table(ROOT, 'g1_boundary_reachability'))
    print(config.RECENT_HISTORY_TITLE)
    display(manifest.load_current_table(ROOT, 'recent4_stage_history'))

,현재 실행 기준 상태
assignments,NOT_GENERATED_BLOCKED
scenario_summary,NOT_GENERATED_BLOCKED
boundary_evidence,NOT_GENERATED_BLOCKED
coalition_table,NOT_GENERATED_BLOCKED
simple_rule_comparison,NOT_GENERATED_BLOCKED
criterion_removal_sensitivity,NOT_GENERATED_BLOCKED
criteria_activation_crosstab,NOT_GENERATED_BLOCKED
criterion_boundary_pass_counts,NOT_GENERATED_BLOCKED
g1_boundary_reachability,NOT_GENERATED_BLOCKED
ppi_stage_divergence,NOT_GENERATED_BLOCKED


점검단계 미산출 — BLOCKED_MISSING_PARAMETERS. 가중치·경계·lambda를 임의로 채우지 않았다.


## 9. 지역 맥락 — EIS 창원시 제조업(업종 점수에 미포함)

In [11]:
print(config.EIS_SCOPE_NOTE)
display(bundle['eis_context'][['quarter', 'eis_changwon_manufacturing', 'eis_manufacturing_yoy_pct']].round(3))

EIS는 창원시 전체 제조업 고용보험 피보험자 수이며 국가산단 업종×분기 패널과 모집단·공간단위가 다르다. 업종별 점수나 점검단계에 넣지 않고 지역 맥락으로만 제시한다.


,quarter,eis_changwon_manufacturing,eis_manufacturing_yoy_pct
0,2022Q1,110281,NaN
1,2022Q2,110525,NaN
2,2022Q3,110156,NaN
3,2022Q4,110008,NaN
4,2023Q1,111749,1.331
5,2023Q2,111930,1.271
6,2023Q3,112476,2.106
7,2023Q4,112818,2.554
8,2024Q1,113327,1.412
9,2024Q2,113950,1.805


## 10. 최신분기 진단카드

아래 카드는 `outputs/report/diagnostic_cards_latest.md`와 같은 내용이다.

In [12]:
display(Markdown(bundle['cards_md']))

# 최신분기 업종별 진단카드 (2026Q2)

> ELECTRE TRI-B의 경계 프로파일과 할당 절차를 적용한 다기준 경계분류 시범모형

- 실행 ID: `20260915T074919114280Z-993960af`
- 모형 명세 버전: `changwon-transition-diagnosis-model/1.0.0`
- 기준 패널 입력 해시: `58499908a6af50344a1362c55c840a58cfd80d4373c7098c4c97a9e0271a573b` (원본 바이트 `9eab0a418a59d45db45d451cd60b05233a7a4f1c756a49150fbd3e50c78ddd6a`)
- 실행 시작: 2026-09-15T07:49:19.114280+00:00
- 파라미터 상태: `DRAFT` · 실행 상태: `BLOCKED_MISSING_PARAMETERS`

## 읽는 방법

- 카드는 **데이터가 확인한 사실 / 모형의 점검단계 / 품질·경계 플래그 / 추가 확인 질문 / 현재 자료로 판단 불가**를 구분한다.
- 업종 순서는 최신분기 고용 규모순이며 점검 우선순위가 아니다.
- Q1 상태(S1~S4·N)는 무엇을 확인할지 정하는 라우팅 범주이며 위험 서열이 아니다.
- 생산 YoY는 명목 생산액 기준이다. PPI 조정 생산 YoY는 미확정 매핑에 따른 민감도 값이다.
- 가동률·업체 수·PPI·EIS는 점검단계를 올리거나 내리지 않는다.
- 할당 절차는 pessimistic assignment이다. 높은 경계 b2부터 비교해 outrank하면 우선점검, 그렇지 않고 b1을 outrank하면 추가확인, 둘 다 아니면 관찰로 배정한다.

## 모형 점검단계: 산출하지 않음

실행 상태 `BLOCKED_MISSING_PARAMETERS` — 가중치·경계(b1·b2)·lambda·delta_emp·require_employment_evidence가 등록되지 않아 ELECTRE 점검단계를 만들지 않았다. 아래 카드에는 점검단계가 없으며, 비어 있는 값을 임의로 채우지 않았다.

- 비어 있는 파라미터 필드 수: 45
- 비어 있는 등록 필드 수: 4
- 비어 있는 승인 필드 수: 8
- 전체 목록: `outputs/tables/model_validation_report.json`의 `parameter_validation`
- canonical 위치에 과거 ELECTRE 산출물이 있어도 현재 결과가 아니다. 현재 실행 여부는 `outputs/current_run_manifest.json`로 확인한다.

## 지역 맥락(EIS, 창원시 전체 제조업)

EIS는 창원시 전체 제조업 고용보험 피보험자 수이며 국가산단 업종×분기 패널과 모집단·공간단위가 다르다. 업종별 점수나 점검단계에 넣지 않고 지역 맥락으로만 제시한다.

| 분기 | EIS 창원시 제조업 피보험자(명) | 전년동기 대비(%) |
|---|---:|---:|
| 2025Q3 | 113,976 | +0.11 |
| 2025Q4 | 114,122 | +0.46 |
| 2026Q1 | 114,572 | +0.44 |
| 2026Q2 | 114,532 | +0.67 |

EIS와 국가산단 고용의 동행 정도로 산단 밖 제조업의 변화 원인을 추론하지 않는다.

## 업종별 카드

### 기계

**1. 데이터가 확인한 사실**

- Q1 상태 `S4` · 명목 생산 YoY -19.55% · 고용 YoY -6.34%
- 고용 58,784명(전년동기 62,763명, 증감 -3,979명)
- 고용감소 절대규모(명) 3,979 · 고용감소 상대규모(%) 6.3397 · 명목 생산감소 정도(%) 19.5474
- 전년동기 대비 고용 하회 연속분기: δ=0.0: 고용 YoY가 기준을 2분기 연속 하회 (최신분기까지 이어진 열린 run) · δ=0.5: 고용 YoY가 기준을 2분기 연속 하회 (최신분기까지 이어진 열린 run) · δ=1.0: 고용 YoY가 기준을 2분기 연속 하회 (최신분기까지 이어진 열린 run)
- 최근 4분기 중 고용 YoY 음수 분기 2/4
- 직전분기 대비 고용 -104명
- 가동업체 1,458개(전년동기 대비 +5개, 직전분기 대비 -4개) · 업체당 고용 40.3명
- 가동률 80.6% (공식값: op_rate_official) · 전년동기 대비 -4.9%p (SAME_SOURCE)
- 고용비중 51.19% · 비례배분 기준선 대비 초과증감 -1,697.3명(원인 아님)
- 최근 상태 전환 `S4->S4`(관측 경로이며 악화·회복 판정이 아님) · 동일 Q1 상태 연속 2분기

**2. 모형의 점검단계**

- 점검단계 미산출 — `BLOCKED_MISSING_PARAMETERS`
- 후속 행동 검토 상태: `NOT_REVIEWED`

**3. 품질·경계 플래그**

- 판정가능 여부 `COMPLETE` · Q1 라우팅 `ROUTABLE_S4` · threshold 플래그 `STABLE_2_0`
- PPI 조정 생산 YoY -20.55%(후보 1개) · PPI 후보 적용 시 Q1 상태 집합 S4 · 모든 PPI 후보에서 명목 생산 방향 유지 · 매핑 미확정(confirmed=False)
- PPI 매핑 검토유형: 단일 후보·근접 분류 · 확정 상태 `NOT_CONFIRMED`
- 가동업체 수 전년동기 대비 변화(firm_count_changed_yoy): 있음 — 표시용이며 고용 변화의 원인 판단이 아님
- 가동업체 10개 이하(small_firm_count_flag): 해당 없음

**4. 추가 확인 질문**

- 수주·가동·기업 수·휴업·고용조정 가능성을 질문으로 확인한다(원인 후보이며 입증된 원인이 아님).

**5. 현재 자료로 판단 불가**

- 고용·생산 변화의 원인(자동화, 업체 증감, 산업 여건, 고용조정 등)은 현재 집계자료로 확인할 수 없다.
- 특정 지원정책의 필요성이나 효과는 현재 자료로 판단할 수 없다.
- 점검단계는 확인 순서를 정하는 분류이며 미래 상태가 나타날 확률을 뜻하지 않는다.

### 전기전자

**1. 데이터가 확인한 사실**

- Q1 상태 `S2` · 명목 생산 YoY +2.70% · 고용 YoY -0.41%
- 고용 27,979명(전년동기 28,095명, 증감 -116명)
- 고용감소 절대규모(명) 116 · 고용감소 상대규모(%) 0.4129 · 명목 생산감소 정도(%) 0.0000
- 전년동기 대비 고용 하회 연속분기: δ=0.0: 고용 YoY가 기준을 6분기 연속 하회 (최신분기까지 이어진 열린 run) · δ=0.5: 고용 YoY가 기준을 하회하지 않음(0분기) · δ=1.0: 고용 YoY가 기준을 하회하지 않음(0분기)
- 최근 4분기 중 고용 YoY 음수 분기 4/4
- 직전분기 대비 고용 +35명 (고용 YoY 음수 구간 중 직전분기 대비 증가)
- 가동업체 590개(전년동기 대비 +19개, 직전분기 대비 +1개) · 업체당 고용 47.4명
- 가동률 76.7% (공식값: op_rate_official) · 전년동기 대비 -0.4%p (SAME_SOURCE)
- 고용비중 24.37% · 비례배분 기준선 대비 초과증감 +905.4명(원인 아님)
- 최근 상태 전환 `S4->S2`(관측 경로이며 악화·회복 판정이 아님) · 동일 Q1 상태 연속 1분기

**2. 모형의 점검단계**

- 점검단계 미산출 — `BLOCKED_MISSING_PARAMETERS`
- 후속 행동 검토 상태: `NOT_REVIEWED`

**3. 품질·경계 플래그**

- 판정가능 여부 `COMPLETE` · Q1 라우팅 `ROUTABLE_S2` · threshold 플래그 `SENSITIVE_0_5`
- PPI 조정 생산 YoY -15.11%~-5.17%(후보 2개) · PPI 후보 적용 시 Q1 상태 집합 S4 · 모든 PPI 후보에서 명목 생산 방향과의 불일치 · 매핑 미확정(confirmed=False)
- PPI 매핑 검토유형: 복수 후보·범위 제시 · 확정 상태 `NOT_CONFIRMED`
- 가동업체 수 전년동기 대비 변화(firm_count_changed_yoy): 있음 — 표시용이며 고용 변화의 원인 판단이 아님
- 가동업체 10개 이하(small_firm_count_flag): 해당 없음

**4. 추가 확인 질문**

- 자동화·외주화·생산성·인력부족·직무구조 변화 가능성을 질문으로 확인한다(원인 후보이며 입증된 원인이 아님).
- PPI 후보 적용 시 명목 생산 방향과의 불일치가 있다: 가격 변화와 물량 변화의 구분을 원자료·현장 확인으로 점검한다(매핑 미확정, 판정 변경 없음).
- 가동률 전년동기 차이의 방향이 명목 생산 YoY 방향과 다르다: 설비·제품구성·재고 변화 여부를 확인한다(판정 변경 없음).
- 고용 YoY는 음수이지만 직전분기 대비 고용은 증가했다: 계절성·일시 요인 여부를 확인한다.
- Q1 상태가 0% 경계에 가깝다(threshold 0.5%에서 N): 상태 해석에 유의한다.

**5. 현재 자료로 판단 불가**

- 고용·생산 변화의 원인(자동화, 업체 증감, 산업 여건, 고용조정 등)은 현재 집계자료로 확인할 수 없다.
- 특정 지원정책의 필요성이나 효과는 현재 자료로 판단할 수 없다.
- 점검단계는 확인 순서를 정하는 분류이며 미래 상태가 나타날 확률을 뜻하지 않는다.

### 운송장비

**1. 데이터가 확인한 사실**

- Q1 상태 `S2` · 명목 생산 YoY +19.64% · 고용 YoY -0.78%
- 고용 16,308명(전년동기 16,437명, 증감 -129명)
- 고용감소 절대규모(명) 129 · 고용감소 상대규모(%) 0.7848 · 명목 생산감소 정도(%) 0.0000
- 전년동기 대비 고용 하회 연속분기: δ=0.0: 고용 YoY가 기준을 6분기 연속 하회 (최신분기까지 이어진 열린 run) · δ=0.5: 고용 YoY가 기준을 6분기 연속 하회 (최신분기까지 이어진 열린 run) · δ=1.0: 고용 YoY가 기준을 하회하지 않음(0분기)
- 최근 4분기 중 고용 YoY 음수 분기 4/4
- 직전분기 대비 고용 -67명
- 가동업체 245개(전년동기 대비 +1개, 직전분기 대비 -1개) · 업체당 고용 66.6명
- 가동률 92.3% (공식값: op_rate_official) · 전년동기 대비 +1.0%p (SAME_SOURCE)
- 고용비중 14.20% · 비례배분 기준선 대비 초과증감 +468.5명(원인 아님)
- 최근 상태 전환 `S2->S2`(관측 경로이며 악화·회복 판정이 아님) · 동일 Q1 상태 연속 2분기

**2. 모형의 점검단계**

- 점검단계 미산출 — `BLOCKED_MISSING_PARAMETERS`
- 후속 행동 검토 상태: `NOT_REVIEWED`

**3. 품질·경계 플래그**

- 판정가능 여부 `COMPLETE` · Q1 라우팅 `ROUTABLE_S2` · threshold 플래그 `SENSITIVE_1_0`
- PPI 조정 생산 YoY +17.24%(후보 1개) · PPI 후보 적용 시 Q1 상태 집합 S2 · 모든 PPI 후보에서 명목 생산 방향 유지 · 매핑 미확정(confirmed=False)
- PPI 매핑 검토유형: 단일 후보·범위 주의 · 확정 상태 `NOT_CONFIRMED`
- 가동업체 수 전년동기 대비 변화(firm_count_changed_yoy): 있음 — 표시용이며 고용 변화의 원인 판단이 아님
- 가동업체 10개 이하(small_firm_count_flag): 해당 없음

**4. 추가 확인 질문**

- 자동화·외주화·생산성·인력부족·직무구조 변화 가능성을 질문으로 확인한다(원인 후보이며 입증된 원인이 아님).
- Q1 상태가 0% 경계에 가깝다(threshold 1.0%에서 N): 상태 해석에 유의한다.

**5. 현재 자료로 판단 불가**

- 고용·생산 변화의 원인(자동화, 업체 증감, 산업 여건, 고용조정 등)은 현재 집계자료로 확인할 수 없다.
- 특정 지원정책의 필요성이나 효과는 현재 자료로 판단할 수 없다.
- 점검단계는 확인 순서를 정하는 분류이며 미래 상태가 나타날 확률을 뜻하지 않는다.

### 철강

**1. 데이터가 확인한 사실**

- Q1 상태 `S2` · 명목 생산 YoY +32.65% · 고용 YoY -1.14%
- 고용 9,689명(전년동기 9,801명, 증감 -112명)
- 고용감소 절대규모(명) 112 · 고용감소 상대규모(%) 1.1427 · 명목 생산감소 정도(%) 0.0000
- 전년동기 대비 고용 하회 연속분기: δ=0.0: 고용 YoY가 기준을 4분기 연속 하회 (최신분기까지 이어진 열린 run) · δ=0.5: 고용 YoY가 기준을 4분기 연속 하회 (최신분기까지 이어진 열린 run) · δ=1.0: 고용 YoY가 기준을 1분기 연속 하회 (최신분기까지 이어진 열린 run)
- 최근 4분기 중 고용 YoY 음수 분기 4/4
- 직전분기 대비 고용 +14명 (고용 YoY 음수 구간 중 직전분기 대비 증가)
- 가동업체 74개(전년동기 대비 -1개, 직전분기 대비 -1개) · 업체당 고용 130.9명
- 가동률 63.7% (공식값: op_rate_official) · 전년동기 대비 +0.2%p (SAME_SOURCE)
- 고용비중 8.44% · 비례배분 기준선 대비 초과증감 +244.3명(원인 아님)
- 최근 상태 전환 `S2->S2`(관측 경로이며 악화·회복 판정이 아님) · 동일 Q1 상태 연속 4분기

**2. 모형의 점검단계**

- 점검단계 미산출 — `BLOCKED_MISSING_PARAMETERS`
- 후속 행동 검토 상태: `NOT_REVIEWED`

**3. 품질·경계 플래그**

- 판정가능 여부 `COMPLETE` · Q1 라우팅 `ROUTABLE_S2` · threshold 플래그 `SENSITIVE_2_0`
- PPI 조정 생산 YoY +25.28%(후보 1개) · PPI 후보 적용 시 Q1 상태 집합 S2 · 모든 PPI 후보에서 명목 생산 방향 유지 · 매핑 미확정(confirmed=False)
- PPI 매핑 검토유형: 단일 후보·분류단계 민감 · 확정 상태 `NOT_CONFIRMED`
- 가동업체 수 전년동기 대비 변화(firm_count_changed_yoy): 있음 — 표시용이며 고용 변화의 원인 판단이 아님
- 가동업체 10개 이하(small_firm_count_flag): 해당 없음

**4. 추가 확인 질문**

- 자동화·외주화·생산성·인력부족·직무구조 변화 가능성을 질문으로 확인한다(원인 후보이며 입증된 원인이 아님).
- 고용 YoY는 음수이지만 직전분기 대비 고용은 증가했다: 계절성·일시 요인 여부를 확인한다.
- Q1 상태가 0% 경계에 가깝다(threshold 2.0%에서 N): 상태 해석에 유의한다.

**5. 현재 자료로 판단 불가**

- 고용·생산 변화의 원인(자동화, 업체 증감, 산업 여건, 고용조정 등)은 현재 집계자료로 확인할 수 없다.
- 특정 지원정책의 필요성이나 효과는 현재 자료로 판단할 수 없다.
- 점검단계는 확인 순서를 정하는 분류이며 미래 상태가 나타날 확률을 뜻하지 않는다.

### 음식료

**1. 데이터가 확인한 사실**

- Q1 상태 `S2` · 명목 생산 YoY +0.87% · 고용 YoY -3.79%
- 고용 635명(전년동기 660명, 증감 -25명)
- 고용감소 절대규모(명) 25 · 고용감소 상대규모(%) 3.7879 · 명목 생산감소 정도(%) 0.0000
- 전년동기 대비 고용 하회 연속분기: δ=0.0: 고용 YoY가 기준을 6분기 연속 하회 (최신분기까지 이어진 열린 run) · δ=0.5: 고용 YoY가 기준을 6분기 연속 하회 (최신분기까지 이어진 열린 run) · δ=1.0: 고용 YoY가 기준을 6분기 연속 하회 (최신분기까지 이어진 열린 run)
- 최근 4분기 중 고용 YoY 음수 분기 4/4
- 직전분기 대비 고용 +7명 (고용 YoY 음수 구간 중 직전분기 대비 증가)
- 가동업체 4개(전년동기 대비 0개, 직전분기 대비 0개) · 업체당 고용 158.8명
- 가동률 92.3% (공식값: op_rate_official) · 전년동기 대비 -1.4%p (SAME_SOURCE)
- 고용비중 0.55% · 비례배분 기준선 대비 초과증감 -1.0명(원인 아님)
- 최근 상태 전환 `S2->S2`(관측 경로이며 악화·회복 판정이 아님) · 동일 Q1 상태 연속 2분기

**2. 모형의 점검단계**

- 점검단계 미산출 — `BLOCKED_MISSING_PARAMETERS`
- 후속 행동 검토 상태: `NOT_REVIEWED`

**3. 품질·경계 플래그**

- 판정가능 여부 `COMPLETE` · Q1 라우팅 `ROUTABLE_S2` · threshold 플래그 `SENSITIVE_1_0`
- PPI 조정 생산 YoY -0.78%(후보 1개) · PPI 후보 적용 시 Q1 상태 집합 S4 · 모든 PPI 후보에서 명목 생산 방향과의 불일치 · 매핑 미확정(confirmed=False)
- PPI 매핑 검토유형: 단일 후보·근접 분류 · 확정 상태 `NOT_CONFIRMED`
- 가동업체 수 전년동기 대비 변화(firm_count_changed_yoy): 없음 — 표시용이며 고용 변화의 원인 판단이 아님
- 가동업체 10개 이하(small_firm_count_flag): 해당
- 가동업체 수가 적어 개별 업체 수준 변화가 업종 집계에 크게 반영될 수 있음. 현재 집계자료만으로 업체 수 변화가 고용 변화의 원인이라고 판단할 수 없음.

**4. 추가 확인 질문**

- 자동화·외주화·생산성·인력부족·직무구조 변화 가능성을 질문으로 확인한다(원인 후보이며 입증된 원인이 아님).
- PPI 후보 적용 시 명목 생산 방향과의 불일치가 있다: 가격 변화와 물량 변화의 구분을 원자료·현장 확인으로 점검한다(매핑 미확정, 판정 변경 없음).
- 가동률 전년동기 차이의 방향이 명목 생산 YoY 방향과 다르다: 설비·제품구성·재고 변화 여부를 확인한다(판정 변경 없음).
- 고용 YoY는 음수이지만 직전분기 대비 고용은 증가했다: 계절성·일시 요인 여부를 확인한다.
- 가동업체 10개 이하 소규모 집계이므로 개별 업체 단위 변화가 있었는지 원자료로 확인한다.
- Q1 상태가 0% 경계에 가깝다(threshold 1.0%에서 N): 상태 해석에 유의한다.

**5. 현재 자료로 판단 불가**

- 고용·생산 변화의 원인(자동화, 업체 증감, 산업 여건, 고용조정 등)은 현재 집계자료로 확인할 수 없다.
- 특정 지원정책의 필요성이나 효과는 현재 자료로 판단할 수 없다.
- 점검단계는 확인 순서를 정하는 분류이며 미래 상태가 나타날 확률을 뜻하지 않는다.

### 석유화학

**1. 데이터가 확인한 사실**

- Q1 상태 `S3` · 명목 생산 YoY -6.73% · 고용 YoY +9.56%
- 고용 619명(전년동기 565명, 증감 +54명)
- 고용감소 절대규모(명) 0 · 고용감소 상대규모(%) 0.0000 · 명목 생산감소 정도(%) 6.7298
- 전년동기 대비 고용 하회 연속분기: δ=0.0: 고용 YoY가 기준을 하회하지 않음(0분기) · δ=0.5: 고용 YoY가 기준을 하회하지 않음(0분기) · δ=1.0: 고용 YoY가 기준을 하회하지 않음(0분기)
- 최근 4분기 중 고용 YoY 음수 분기 1/4
- 직전분기 대비 고용 -3명
- 가동업체 45개(전년동기 대비 +4개, 직전분기 대비 +2개) · 업체당 고용 13.8명
- 가동률 76.3% (공식값: op_rate_official) · 전년동기 대비 -3.3%p (SAME_SOURCE)
- 고용비중 0.54% · 비례배분 기준선 대비 초과증감 +74.5명(원인 아님)
- 최근 상태 전환 `S4->S3`(관측 경로이며 악화·회복 판정이 아님) · 동일 Q1 상태 연속 1분기

**2. 모형의 점검단계**

- 점검단계 미산출 — `BLOCKED_MISSING_PARAMETERS`
- 후속 행동 검토 상태: `NOT_REVIEWED`

**3. 품질·경계 플래그**

- 판정가능 여부 `COMPLETE` · Q1 라우팅 `ROUTABLE_S3` · threshold 플래그 `STABLE_2_0`
- PPI 조정 생산 YoY -45.54%~-21.72%(후보 2개) · PPI 후보 적용 시 Q1 상태 집합 S3 · 모든 PPI 후보에서 명목 생산 방향 유지 · 매핑 미확정(confirmed=False)
- PPI 매핑 검토유형: 복수 후보·범위 제시 · 확정 상태 `NOT_CONFIRMED`
- 가동업체 수 전년동기 대비 변화(firm_count_changed_yoy): 있음 — 표시용이며 고용 변화의 원인 판단이 아님
- 가동업체 10개 이하(small_firm_count_flag): 해당 없음

**4. 추가 확인 질문**

- 선제채용·신규기업·고용조정 시차·생산 시차 가능성을 질문으로 확인한다(원인 후보이며 입증된 원인이 아님).

**5. 현재 자료로 판단 불가**

- 고용·생산 변화의 원인(자동화, 업체 증감, 산업 여건, 고용조정 등)은 현재 집계자료로 확인할 수 없다.
- 특정 지원정책의 필요성이나 효과는 현재 자료로 판단할 수 없다.
- 점검단계는 확인 순서를 정하는 분류이며 미래 상태가 나타날 확률을 뜻하지 않는다.

### 목재종이

**1. 데이터가 확인한 사실**

- Q1 상태 `S4` · 명목 생산 YoY -12.90% · 고용 YoY -10.08%
- 고용 428명(전년동기 476명, 증감 -48명)
- 고용감소 절대규모(명) 48 · 고용감소 상대규모(%) 10.0840 · 명목 생산감소 정도(%) 12.8971
- 전년동기 대비 고용 하회 연속분기: δ=0.0: 고용 YoY가 기준을 2분기 연속 하회 (최신분기까지 이어진 열린 run) · δ=0.5: 고용 YoY가 기준을 2분기 연속 하회 (최신분기까지 이어진 열린 run) · δ=1.0: 고용 YoY가 기준을 2분기 연속 하회 (최신분기까지 이어진 열린 run)
- 최근 4분기 중 고용 YoY 음수 분기 2/4
- 직전분기 대비 고용 0명
- 가동업체 44개(전년동기 대비 -1개, 직전분기 대비 -2개) · 업체당 고용 9.7명
- 가동률 74.7% (공식값: op_rate_official) · 전년동기 대비 +6.0%p (SAME_SOURCE)
- 고용비중 0.37% · 비례배분 기준선 대비 초과증감 -30.7명(원인 아님)
- 최근 상태 전환 `S4->S4`(관측 경로이며 악화·회복 판정이 아님) · 동일 Q1 상태 연속 2분기

**2. 모형의 점검단계**

- 점검단계 미산출 — `BLOCKED_MISSING_PARAMETERS`
- 후속 행동 검토 상태: `NOT_REVIEWED`

**3. 품질·경계 플래그**

- 판정가능 여부 `COMPLETE` · Q1 라우팅 `ROUTABLE_S4` · threshold 플래그 `STABLE_2_0`
- PPI 조정 생산 YoY -15.29%(후보 1개) · PPI 후보 적용 시 Q1 상태 집합 S4 · 모든 PPI 후보에서 명목 생산 방향 유지 · 매핑 미확정(confirmed=False)
- PPI 매핑 검토유형: 단일 후보·근접 분류 · 확정 상태 `NOT_CONFIRMED`
- 가동업체 수 전년동기 대비 변화(firm_count_changed_yoy): 있음 — 표시용이며 고용 변화의 원인 판단이 아님
- 가동업체 10개 이하(small_firm_count_flag): 해당 없음

**4. 추가 확인 질문**

- 수주·가동·기업 수·휴업·고용조정 가능성을 질문으로 확인한다(원인 후보이며 입증된 원인이 아님).
- 가동률 전년동기 차이의 방향이 명목 생산 YoY 방향과 다르다: 설비·제품구성·재고 변화 여부를 확인한다(판정 변경 없음).

**5. 현재 자료로 판단 불가**

- 고용·생산 변화의 원인(자동화, 업체 증감, 산업 여건, 고용조정 등)은 현재 집계자료로 확인할 수 없다.
- 특정 지원정책의 필요성이나 효과는 현재 자료로 판단할 수 없다.
- 점검단계는 확인 순서를 정하는 분류이며 미래 상태가 나타날 확률을 뜻하지 않는다.

### 비금속

**1. 데이터가 확인한 사실**

- Q1 상태 `S2` · 명목 생산 YoY +45.67% · 고용 YoY -1.08%
- 고용 183명(전년동기 185명, 증감 -2명)
- 고용감소 절대규모(명) 2 · 고용감소 상대규모(%) 1.0811 · 명목 생산감소 정도(%) 0.0000
- 전년동기 대비 고용 하회 연속분기: δ=0.0: 고용 YoY가 기준을 최소 18분기 연속 하회 (분석 시작분기부터 이어진 좌절단 관측; 분석창 전체 길이 도달; 최신분기까지 이어진 열린 run) · δ=0.5: 고용 YoY가 기준을 최소 18분기 연속 하회 (분석 시작분기부터 이어진 좌절단 관측; 분석창 전체 길이 도달; 최신분기까지 이어진 열린 run) · δ=1.0: 고용 YoY가 기준을 최소 18분기 연속 하회 (분석 시작분기부터 이어진 좌절단 관측; 분석창 전체 길이 도달; 최신분기까지 이어진 열린 run)
- 최근 4분기 중 고용 YoY 음수 분기 4/4
- 직전분기 대비 고용 +1명 (고용 YoY 음수 구간 중 직전분기 대비 증가)
- 가동업체 4개(전년동기 대비 -1개, 직전분기 대비 -1개) · 업체당 고용 45.8명
- 가동률 57.3% (공식값: op_rate_official) · 전년동기 대비 +14.1%p (SAME_SOURCE)
- 고용비중 0.16% · 비례배분 기준선 대비 초과증감 +4.7명(원인 아님)
- 최근 상태 전환 `S2->S2`(관측 경로이며 악화·회복 판정이 아님) · 동일 Q1 상태 연속 3분기

**2. 모형의 점검단계**

- 점검단계 미산출 — `BLOCKED_MISSING_PARAMETERS`
- 후속 행동 검토 상태: `NOT_REVIEWED`

**3. 품질·경계 플래그**

- 판정가능 여부 `COMPLETE` · Q1 라우팅 `ROUTABLE_S2` · threshold 플래그 `SENSITIVE_2_0`
- PPI 조정 생산 YoY +40.86%(후보 1개) · PPI 후보 적용 시 Q1 상태 집합 S2 · 모든 PPI 후보에서 명목 생산 방향 유지 · 매핑 미확정(confirmed=False)
- PPI 매핑 검토유형: 단일 후보·근접 분류 · 확정 상태 `NOT_CONFIRMED`
- 가동업체 수 전년동기 대비 변화(firm_count_changed_yoy): 있음 — 표시용이며 고용 변화의 원인 판단이 아님
- 가동업체 10개 이하(small_firm_count_flag): 해당
- 가동업체 수가 적어 개별 업체 수준 변화가 업종 집계에 크게 반영될 수 있음. 현재 집계자료만으로 업체 수 변화가 고용 변화의 원인이라고 판단할 수 없음.

**4. 추가 확인 질문**

- 자동화·외주화·생산성·인력부족·직무구조 변화 가능성을 질문으로 확인한다(원인 후보이며 입증된 원인이 아님).
- 고용 YoY는 음수이지만 직전분기 대비 고용은 증가했다: 계절성·일시 요인 여부를 확인한다.
- 가동업체 10개 이하 소규모 집계이므로 개별 업체 단위 변화가 있었는지 원자료로 확인한다.
- Q1 상태가 0% 경계에 가깝다(threshold 2.0%에서 N): 상태 해석에 유의한다.

**5. 현재 자료로 판단 불가**

- 고용·생산 변화의 원인(자동화, 업체 증감, 산업 여건, 고용조정 등)은 현재 집계자료로 확인할 수 없다.
- 특정 지원정책의 필요성이나 효과는 현재 자료로 판단할 수 없다.
- 점검단계는 확인 순서를 정하는 분류이며 미래 상태가 나타날 확률을 뜻하지 않는다.

### 기타

**1. 데이터가 확인한 사실**

- Q1 상태 `S1` · 명목 생산 YoY +3.03% · 고용 YoY +17.12%
- 고용 171명(전년동기 146명, 증감 +25명)
- 고용감소 절대규모(명) 0 · 고용감소 상대규모(%) 0.0000 · 명목 생산감소 정도(%) 0.0000
- 전년동기 대비 고용 하회 연속분기: δ=0.0: 고용 YoY가 기준을 하회하지 않음(0분기) · δ=0.5: 고용 YoY가 기준을 하회하지 않음(0분기) · δ=1.0: 고용 YoY가 기준을 하회하지 않음(0분기)
- 최근 4분기 중 고용 YoY 음수 분기 2/4
- 직전분기 대비 고용 +2명
- 가동업체 25개(전년동기 대비 -2개, 직전분기 대비 -1개) · 업체당 고용 6.8명
- 가동률 69.1% (공식값: op_rate_official) · 전년동기 대비 +1.6%p (SAME_SOURCE)
- 고용비중 0.15% · 비례배분 기준선 대비 초과증감 +30.3명(원인 아님)
- 최근 상태 전환 `S1->S1`(관측 경로이며 악화·회복 판정이 아님) · 동일 Q1 상태 연속 2분기

**2. 모형의 점검단계**

- 점검단계 미산출 — `BLOCKED_MISSING_PARAMETERS`
- 후속 행동 검토 상태: `NOT_REVIEWED`

**3. 품질·경계 플래그**

- 판정가능 여부 `COMPLETE` · Q1 라우팅 `ROUTABLE_S1` · threshold 플래그 `STABLE_2_0`
- 비교불가(NO_MAPPING_CANDIDATE) · 매핑 미확정(confirmed=False)
- PPI 매핑 검토유형: 매핑 불가 · 확정 상태 `NOT_CONFIRMED`
- 가동업체 수 전년동기 대비 변화(firm_count_changed_yoy): 있음 — 표시용이며 고용 변화의 원인 판단이 아님
- 가동업체 10개 이하(small_firm_count_flag): 해당 없음
- `REQUIRES_DISAGGREGATION` — 잔여 범주 — 단일 업종으로 해석하거나 특정 업종 지원사업에 직접 연결할 수 없음. 구성 업종 확인 후 라우팅 필요.

**4. 추가 확인 질문**

- 미충원·숙련수요·증가 지속가능성을 확인한다(현재 자료로 단정하지 않음).
- 잔여 범주 — 단일 업종으로 해석하거나 특정 업종 지원사업에 직접 연결할 수 없음. 구성 업종 확인 후 라우팅 필요.

**5. 현재 자료로 판단 불가**

- 고용·생산 변화의 원인(자동화, 업체 증감, 산업 여건, 고용조정 등)은 현재 집계자료로 확인할 수 없다.
- 특정 지원정책의 필요성이나 효과는 현재 자료로 판단할 수 없다.
- 점검단계는 확인 순서를 정하는 분류이며 미래 상태가 나타날 확률을 뜻하지 않는다.

### 섬유의복

**1. 데이터가 확인한 사실**

- Q1 상태 `N` · 명목 생산 YoY -12.32% · 고용 YoY 0.00%
- 고용 34명(전년동기 34명, 증감 0명)
- 고용감소 절대규모(명) 0 · 고용감소 상대규모(%) 0.0000 · 명목 생산감소 정도(%) 12.3197
- 전년동기 대비 고용 하회 연속분기: δ=0.0: 고용 YoY가 기준을 하회하지 않음(0분기) · δ=0.5: 고용 YoY가 기준을 하회하지 않음(0분기) · δ=1.0: 고용 YoY가 기준을 하회하지 않음(0분기)
- 최근 4분기 중 고용 YoY 음수 분기 1/4
- 직전분기 대비 고용 0명
- 가동업체 8개(전년동기 대비 +1개, 직전분기 대비 0개) · 업체당 고용 4.2명
- 가동률 64.0% (공식값: op_rate_official) · 전년동기 대비 -9.0%p (SAME_SOURCE)
- 고용비중 0.03% · 비례배분 기준선 대비 초과증감 +1.2명(원인 아님)
- 최근 상태 전환 `N->N`(관측 경로이며 악화·회복 판정이 아님) · 동일 Q1 상태 연속기간 해당 없음(S1~S4가 아님)

**2. 모형의 점검단계**

- 점검단계 미산출 — `BLOCKED_MISSING_PARAMETERS`
- 후속 행동 검토 상태: `NOT_REVIEWED`

**3. 품질·경계 플래그**

- 판정가능 여부 `COMPLETE` · Q1 라우팅 `NEUTRAL_REVIEW` · threshold 플래그 `EXACT_ZERO`
- PPI 조정 생산 YoY -15.17%(후보 1개) · PPI 후보 적용 시 Q1 상태 집합 N · 모든 PPI 후보에서 명목 생산 방향 유지 · 매핑 미확정(confirmed=False)
- PPI 매핑 검토유형: 단일 후보·분류 선택 · 확정 상태 `NOT_CONFIRMED`
- 가동업체 수 전년동기 대비 변화(firm_count_changed_yoy): 있음 — 표시용이며 고용 변화의 원인 판단이 아님
- 가동업체 10개 이하(small_firm_count_flag): 해당
- 가동업체 수가 적어 개별 업체 수준 변화가 업종 집계에 크게 반영될 수 있음. 현재 집계자료만으로 업체 수 변화가 고용 변화의 원인이라고 판단할 수 없음.

**4. 추가 확인 질문**

- 원자료상 정확한 0인지 반올림·개정 영향인지, 0이 아닌 다른 축의 방향과 규모, 0.5·1·2% 민감도에서의 해석 변화를 확인한다.
- 가동업체 10개 이하 소규모 집계이므로 개별 업체 단위 변화가 있었는지 원자료로 확인한다.

**5. 현재 자료로 판단 불가**

- 고용·생산 변화의 원인(자동화, 업체 증감, 산업 여건, 고용조정 등)은 현재 집계자료로 확인할 수 없다.
- 특정 지원정책의 필요성이나 효과는 현재 자료로 판단할 수 없다.
- 점검단계는 확인 순서를 정하는 분류이며 미래 상태가 나타날 확률을 뜻하지 않는다.

## 판정불가 행(데이터 검토)

최신분기에는 판정불가 행이 없다. 전체 패널의 판정불가 행 20건은 `outputs/tables/electre_eligibility_audit.csv`에 원인과 함께 보존했다. 판정불가는 관찰보다 낮은 단계가 아니라 별도 데이터 검토 대상이다.

## 단계별 후속 행동

상태 `POLICY_DECISION_REQUIRED` — 담당 역할·처리기한·필수 확인·산출물은 실제 기관이 정해야 하며 이 카드에서 임의로 지정하지 않았다(`config/stage_actions.template.yaml`).
